In [0]:
%sql
CREATE OR REPLACE TEMPORARY FUNCTION tcp_ping(
    host STRING,
    port INT,
    timeout_seconds DOUBLE
)
RETURNS STRING
LANGUAGE PYTHON
AS $$
import socket
import time

started = time.perf_counter()

try:
    with socket.create_connection(
        (host, port),
        timeout=timeout_seconds
    ):
        elapsed_ms = round((time.perf_counter() - started) * 1000, 2)
        return f"CONNECTED in {elapsed_ms} ms"

except Exception as error:
    elapsed_ms = round((time.perf_counter() - started) * 1000, 2)
    return f"FAILED after {elapsed_ms} ms: {type(error).__name__}: {error}"
$$;

In [0]:
from pyspark.sql import functions as F

number_of_tests = 8

result = (
    spark.range(number_of_tests)
    .repartition(number_of_tests)
    .select(
        F.col("id").alias("test_id"),
        F.spark_partition_id().alias("partition_id"),
        F.expr(
            "tcp_ping('www.google.com', 443, 3.0)"
        ).alias("internet_test"),
    )
)

display(result)

test_id,partition_id,internet_test
2,0,CONNECTED in 16.13 ms
5,0,CONNECTED in 12.02 ms
0,2,CONNECTED in 19.5 ms
1,2,CONNECTED in 17.24 ms
7,4,CONNECTED in 22.07 ms
4,5,CONNECTED in 22.25 ms
3,6,CONNECTED in 22.54 ms
6,7,CONNECTED in 15.3 ms
